# **YOLOv8n**

In [ ]:
!pip install -q ultralytics torchvision tqdma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.8 MB/s eta 0:00:00


In [ ]:
import zipfile
import shutil
from pathlib import Path

import yaml
import torch
from google.colab import drive
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


1. Mounting Drive

In [ ]:
drive.mount("/content/drive")

DRIVE_DIR     = Path("/content/drive/MyDrive/PPE_Compliance_Project")
DRIVE_RESULTS = DRIVE_DIR / "yolov8n_results"
METRICS_LOG   = DRIVE_DIR / "yolov8n_metrics_summary.txt"
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


2. Extracting the Dataset

In [ ]:
ZIP_PATH = Path("/content/Roboflow.zip")
DATA_DIR = Path("/content/ppe_data")

if not DATA_DIR.exists():
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)
    print(f"Extracted to: {DATA_DIR}")
else:
    print("Dataset already present, skipping extraction.")

Extracting dataset...
Extracted to: /content/ppe_data


3. Locating data.yaml

In [ ]:
DATA_YAML = DATA_DIR / "results_yolov8n_100e" / "kaggle" / "working" / "ppe_data.yaml"

if not DATA_YAML.exists():
    raise FileNotFoundError(f"Expected data.yaml not found at {DATA_YAML}")

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

# The dataset root seems to be DATA_DIR / "css-data" based on the file listing
cfg["path"]  = str(DATA_DIR / "css-data")
cfg["train"] = "train/images"
cfg["val"]   = "valid/images"
cfg["test"]  = "test/images"

with open(DATA_YAML, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f"data.yaml sanitized: {DATA_YAML}")

data.yaml sanitized: /content/ppe_data/results_yolov8n_100e/kaggle/working/ppe_data.yaml


4. Sanity checking

In [ ]:
def verify_split(split_name: str, img_subdir: str, lbl_subdir: str):
    img_dir = DATA_DIR / "css-data" / split_name / img_subdir
    lbl_dir = DATA_DIR / "css-data" / split_name / lbl_subdir

    if not img_dir.exists():
        raise RuntimeError(f"Missing image directory: {img_dir}")
    if not lbl_dir.exists():
        raise RuntimeError(f"Missing label directory: {lbl_dir}")

    images = list(img_dir.glob("*.[jp][pn]g")) + list(img_dir.glob("*.jpeg"))
    labels = list(lbl_dir.glob("*.txt"))

    if not images:
        raise RuntimeError(f"No images found in {img_dir}")
    if not labels:
        raise RuntimeError(f"No label files found in {lbl_dir}")

    print(f"  {split_name:<6}  {len(images):>5} images  {len(labels):>5} labels")

print("Verifying splits:")
for split in ("train", "valid", "test"):
    verify_split(split, "images", "labels")

Verifying splits:
  train    2605 images   2605 labels
  valid     114 images    114 labels
  test       82 images     82 labels


5. Training

In [ ]:
model = YOLO("yolov8n.pt")

try:
    model.train(
        data          = str(DATA_YAML),
        device        = 0,
        imgsz         = 640,
        epochs        = 50,
        patience      = 15,
        batch         = 16,
        lr0           = 1e-3,
        weight_decay  = 5e-4,
        warmup_epochs = 3,
        seed          = 42,
        cls_pw        = 1.0,
        cls           = 2.0,
        close_mosaic  = 10,
        project       = "runs/ppe",
        name          = "yolov8n_v1",
        exist_ok      = True,
    )

except RuntimeError as e:
    if "out of memory" in str(e).lower():
        torch.cuda.empty_cache()
        print(
            "\n[CUDA OOM] GPU memory exhausted.\n"
            "  Reduce batch to 8 or drop imgsz to 416 and retry.\n"
            "  Switch to an A100 runtime if the issue persists."
        )
    raise

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.0, cls_pw=1.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ppe_data/results_yolov8n_100e/kaggle/working/ppe_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_v1, nbs=64, nms=False, opset=None, optimize=False, optimize

6. Evaluating on the test split

In [ ]:
print("\nEvaluating on test split...")
val_results = model.val(
    data   = str(DATA_YAML),
    split  = "test",
    imgsz  = 640,
    device = 0,
)

global_map50    = val_results.box.map50   # float
global_map50_95 = val_results.box.map     # float
per_class_ap50  = val_results.box.ap50
class_names     = model.names


Evaluating on test split...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1556.9±408.7 MB/s, size: 49.3 KB)
val: Scanning /content/ppe_data/css-data/test/labels... 82 images, 8 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 82/82 2.2Kit/s 0.0s
val: New cache created: /content/ppe_data/css-data/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.8it/s 3.3s
                   all         82        760      0.865      0.676      0.732      0.402
               Hardhat         30        110      0.958      0.834      0.886       0.52
                  Mask         16         28      0.978       0.75      0.745      0.437
            NO-Hardhat         25         41      0.813      0.561      0.554      0.241
               NO-Mask         30         7

7. Metrics Report

In [ ]:
divider = "=" * 50

lines = [
    divider,
    "  YOLOv8n PPE Compliance — Test Set Evaluation",
    divider,
    f"  Global mAP50        {global_map50:.4f}",
    f"  Global mAP50-95     {global_map50_95:.4f}",
    "",
    "  Per-Class mAP50",
    "  " + "-" * 35,
]

for idx, name in class_names.items():
    score = per_class_ap50[idx] if idx < len(per_class_ap50) else float("nan")
    lines.append(f"  {name:<28} {score:.4f}")

lines += ["", divider]
report = "\n".join(lines)

print("\n" + report)

with open(METRICS_LOG, "w") as f:
    f.write(report + "\n")

print(f"\nMetrics saved: {METRICS_LOG}")


  YOLOv8n PPE Compliance — Test Set Evaluation
  Global mAP50        0.7324
  Global mAP50-95     0.4022

  Per-Class mAP50
  -----------------------------------
  Hardhat                      0.8864
  Mask                         0.7450
  NO-Hardhat                   0.5536
  NO-Mask                      0.7211
  NO-Safety Vest               0.7839
  Person                       0.7823
  Safety Cone                  0.4238
  Safety Vest                  0.8825
  machinery                    0.8441
  vehicle                      0.7010


Metrics saved: /content/drive/MyDrive/PPE_Compliance_Project/yolov8n_metrics_summary.txt


8. Migrating to Drive

In [ ]:
RUNS_DIR = Path("runs")

if RUNS_DIR.exists():
    if DRIVE_RESULTS.exists():
        shutil.rmtree(DRIVE_RESULTS)

    print(f"Copying artifacts to Drive...")
    shutil.copytree(str(RUNS_DIR), str(DRIVE_RESULTS))
    shutil.rmtree(str(RUNS_DIR))
    print(f"Artifacts at: {DRIVE_RESULTS}")
    print("Local runs/ cleaned up.")
else:
    print("Warning: runs/ not found — nothing to migrate.")

print("\nDone.")

Copying artifacts to Drive...
Artifacts at: /content/drive/MyDrive/PPE_Compliance_Project/yolov8n_results
Local runs/ cleaned up.

Done.
